In [11]:
import pandas as pd

price_path = '/content/수정주가.xlsx'
pbr_path = '/content/PBR.xlsx'
roe_path = '/content/ROE.xlsx'

price_df = pd.read_excel(price_path, header=9, index_col=0)
pbr_df = pd.read_excel(pbr_path, header=9, index_col=0)
roe_df = pd.read_excel(roe_path, header=9, index_col=0).ffill()

# 1. 가장 최근 PBR, ROE 가져오기
latest_pbr = pbr_df.iloc[-1]
latest_roe = roe_df.iloc[-1]

# 2. 모멘텀 계산 (최근 3개월 = 약 60영업일 기준 수익률)
# (오늘 주가 - 3개월 전 주가) / 3개월 전 주가
momentum_3m = (price_df.iloc[-1] - price_df.iloc[-60]) / price_df.iloc[-60]

# 최근 3개월 KOSPI 수익률 5%(임시)
kospi_3m_return = 0.05

df = pd.DataFrame({
    'PBR': latest_pbr,
    'ROE': latest_roe,
    '3개월수익률': momentum_3m
})

# 글자 -> 숫자 변환
df['PBR'] = pd.to_numeric(df['PBR'], errors='coerce')
df['ROE'] = pd.to_numeric(df['ROE'], errors='coerce')
df['3개월수익률'] = pd.to_numeric(df['3개월수익률'], errors='coerce')
df = df.dropna()


# 조건 1: PBR 0보다 크고 1 미만 (좀비 기업 제외 & 가치주)
cond_pbr = (df['PBR'] > 0) & (df['PBR'] < 1.0)

# 조건 2: ROE 10% 이상 (우량주)
cond_roe = df['ROE'] >= 10.0

# 조건 3: 최근 3개월 수익률이 코스피보다 높음 (모멘텀)
cond_mom = df['3개월수익률'] > kospi_3m_return

final_stocks = df[cond_pbr & cond_roe & cond_mom]

# 모멘텀(수익률)이 가장 좋은 순서대로 10개
top_10 = final_stocks.sort_values(by='3개월수익률', ascending=False).head(10)

print("\n[10종목 (PBR<1 + ROE>10% + 코스피 이긴 모멘텀)]")

# 수익률을 퍼센트(%),소수점 2자리
top_10['3개월수익률(%)'] = (top_10['3개월수익률'] * 100).round(2)
display(top_10[['PBR', 'ROE', '3개월수익률(%)']].round(2))


[10종목 (PBR<1 + ROE>10% + 코스피 이긴 모멘텀)]


,PBR,ROE,3개월수익률(%)
삼지전자,0.97,17.09,81.82
베뉴지,0.62,13.09,68.25
일지테크,0.55,26.95,52.24
기산텔레콤,0.86,26.60,43.94
체리부로,0.68,23.39,42.57
원익,0.66,10.99,35.67
우원개발,0.80,24.91,34.36
화신,0.76,12.25,33.96
파커스,0.40,10.04,29.93
케이비아이동국실업,0.32,12.78,29.74
